2.1 模式1：Pydantic
它通过在运行时强制执行类型提示，确保数据的正确性和一致性，是
生产场景首选。
2.1.1 基本使用
需要满足的几个要素：
所有结构化输出的数据模型都必须继承 BaseModel
使用类型提示。Pydantic 支持丰富的字段类型：str 、int、float、List[xxx]、Optional[xxx]等
使用 Field() 添加字段默认值和描述，帮助 LLM 理解字段含义
举例1：

In [11]:
from typing import Optional

from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

load_dotenv(override=True)
DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = "https://api.deepseek.com"
model = init_chat_model(
    model="deepseek-v4-flash",
    model_provider="deepseek",
    api_key=DEEPSEEK_API_KEY,
    base_url=DEEPSEEK_BASE_URL,
    extra_body={"thinking": {"type": "disabled"}}  #关闭思考模式
)

from pydantic import BaseModel, Field


class Person(BaseModel):
    """人物信息"""
    name: str = Field(description="姓名")
    age: int = Field(description="年龄")
    occupation: str = Field(description="职业")


#LangChain 会要求 LLM 的输出必须能填充这些字段。


# 使用with_structured_output即可引导模型进行结构化输出：

# 创建结构化输出的 LLM
structured_llm = model.with_structured_output(Person)
# 调用
result: Person = structured_llm.invoke("张三是一名 30 岁的软件工程师")
print(result)
# result 是 Person 实例
print(result.name)  # "张三"
print(result.age)  # 30
print(result.occupation)  # "软件工程师"

name='张三' age=30 occupation='软件工程师'
张三
30
软件工程师


1.高级特性  Optional 可选字段

In [13]:
from typing import Optional


class Person(BaseModel):
    """人物信息"""
    name: str = Field(description="姓名")
    age: Optional[int] = Field(description="年龄")
    occupation: str = Field(description="职业")


#LangChain 会要求 LLM 的输出必须能填充这些字段。


# 使用with_structured_output即可引导模型进行结构化输出：

# 创建结构化输出的 LLM
structured_llm = model.with_structured_output(Person)
# 调用
result: Person = structured_llm.invoke("张三是一名软件工程师")
print(result)
# result 是 Person 实例
print(result.name)  # "张三"
print(result.age)  # 30
print(result.occupation)  # "软件工程师"

name='张三' age=None occupation='软件工程师'
张三
None
软件工程师


默认值
不同的模型供应商,对于此字段的支持是不同的
- Deepseek 支持


In [14]:

class Person(BaseModel):
    """人物信息"""
    name: str = Field(description="姓名")
    age: int = Field(default=30, description="年龄")
    occupation: str = Field(description="职业")


#LangChain 会要求 LLM 的输出必须能填充这些字段。


# 使用with_structured_output即可引导模型进行结构化输出：

# 创建结构化输出的 LLM
structured_llm = model.with_structured_output(Person)
# 调用
result: Person = structured_llm.invoke("张三是一名软件工程师")
print(result)
# result 是 Person 实例
print(result.name)  # "张三"
print(result.age)  # 30
print(result.occupation)  # "软件工程师"

name='张三' age=30 occupation='软件工程师'
张三
30
软件工程师


情况3：枚举类型
问题：如何限制字段的可选值？
回答：使用枚举。
举例1：

In [18]:
from enum import Enum
from typing import Optional
from pydantic import BaseModel, Field


# 定义你的优先级枚举类
class Priority(str, Enum):
    LOW = "低"
    MEDIUM = "中"
    HIGH = "高"


class CustomerInfo(BaseModel):
    """客户信息"""
    name: str = Field(description="客户姓名")
    phone: str = Field(description="电话号码")
    email: Optional[str] = Field(description="邮箱")
    issue: str = Field(description="问题描述")
    urgency: Priority = Field(description="紧急程度")


# 测试
structured_llm = model.with_structured_output(CustomerInfo)

conversation = """
    客服: 您好，请问有什么可以帮助您？
    客户: 我是王小明，电话 138-1234-5678，我的订单一直没发货，很着急！
    客服: 好的，我帮您查一下
    """
result: CustomerInfo = structured_llm.invoke(f"从以下客服对话中提取客户信息：\n{conversation}")
print(result)
print("\n提取结果：")
print(f"  客户: {result.name}")
print(f"  电话: {result.phone}")
print(f"  邮箱: {result.email or '未提供'}")
print(f"  问题: {result.issue}")
print(f"  紧急程度: {result.urgency.value}")


name='王小明' phone='138-1234-5678' email='null' issue='订单一直没发货' urgency=<Priority.HIGH: '高'>
name='王小明' phone='138-1234-5678' email='null' issue='订单一直没发货' urgency=<Priority.HIGH: '高'>

提取结果：
  客户: 王小明
  电话: 138-1234-5678
  邮箱: null
  问题: 订单一直没发货
  紧急程度: 高


如果嫌单独定义一个
Enum 类太麻烦，也可以直接导入
typing 中的
的值写死。

In [19]:
from typing import Optional, Literal
from pydantic import BaseModel, Field


class CustomerInfo(BaseModel):
    """客户信息"""
    name: str = Field(description="客户姓名")
    phone: str = Field(description="电话号码")
    email: Optional[str] = Field("未提供", description="邮箱")
    issue: str = Field(description="问题描述")
    # 使用 Literal 直接限定字面量值
    urgency: Literal["低", "中", "高"] = Field(description="紧急程度", default="中")


# 测试
structured_llm = model.with_structured_output(CustomerInfo)

conversation = """
客服: 您好，请问有什么可以帮助您？
客户: 我是王小明，电话 138-1234-5678，我的订单一直没发货，很着急！
客服: 好的，我帮您查一下
"""
result = structured_llm.invoke(f"从以下客服对话中提取客户信息：\n{conversation}")
print("\n提取结果：")
print(f"  客户: {result.name}")
print(f"  电话: {result.phone}")
print(f"  邮箱: {result.email}")
print(f"  问题: {result.issue}")
print(f"  紧急程度: {result.urgency}")



提取结果：
  客户: 王小明
  电话: 138-1234-5678
  邮箱: 未提供
  问题: 订单一直没发货
  紧急程度: 高


情况4：列表提取

In [20]:
from typing import List


class Person(BaseModel):
    """人物信息"""
    name: str
    age: int


class PersonList(BaseModel):
    """人物列表信息"""
    people: List[Person]  # 多个 Person 对象


structured_llm = model.with_structured_output(PersonList)
result = structured_llm.invoke("张三 30岁，李四 25岁")
print(result)

people=[Person(name='张三', age=30), Person(name='李四', age=25)]


举例2：产品评论分析

In [21]:
class Review(BaseModel):
    """产品评论"""
    product: str
    rating: int = Field(description="评分 1-5")
    pros: List[str] = Field(description="优点列表")
    cons: List[str] = Field(description="缺点列表")


structured_llm = model.with_structured_output(Review)
review = structured_llm.invoke("""
iPhone 17 很棒！摄像头强大，手感好。但是价格贵，没有充电器。4分。
""")
print(review)

product='iPhone 17' rating=4 pros=['摄像头强大', '手感好'] cons=['价格贵', '没有充电器']


举例3：文档信息提取

In [22]:
class Invoice(BaseModel):
    """发票信息"""
    invoice_number: str = Field(description="发票号")
    date: str = Field(description="日期")
    total_amount: float = Field(description="总金额")
    items: List[str] = Field(description="商品")


# 测试
structured_llm = model.with_structured_output(Invoice)
invoice_text = """
发票号: INV-2024-001
日期: 2024-01-15
总金额: 1299.00
商品: MacBook Pro, AppleCare+
"""
invoice = structured_llm.invoke(f"提取发票信息：{invoice_text}")
print(invoice)

invoice_number='INV-2024-001' date='2024-01-15' total_amount=1299.0 items=['MacBook Pro', 'AppleCare+']


情况5：嵌套结构

In [23]:
from pydantic import BaseModel


class Address(BaseModel):
    """地点描述"""
    city: str
    district: str


class Company(BaseModel):
    """公司信息"""
    name: str
    address: Address  # 嵌套模型


structured_llm = model.with_structured_output(Company)

result = structured_llm.invoke("阿里巴巴在杭州滨江区")
print(result)

name='阿里巴巴' address=Address(city='杭州', district='滨江区')


In [27]:
from pydantic import BaseModel, Field, ValidationError
from typing import List


# 1. 定义嵌套的 Pydantic 模型
class Actor(BaseModel):
    """演员信息"""
    name: str = Field(description="演员姓名")
    role: str = Field(description="饰演的角色")


class Movie(BaseModel):
    """电影信息"""
    title: str = Field(description="电影标题", required=True)
    year: int = Field(description="上映年份", required=True)
    director: str = Field(description="导演", required=True)
    cast: List[Actor] = Field(description="演员列表", required=True)  # 定义列表字段
    rating: float = Field(description="评分", required=True)


# 2. 初始化模型并绑定输出结构
structured_model = model.with_structured_output(Movie)
# 3. 调用模型，直接获取 Movie 实例
response: Movie = structured_model.invoke("请介绍电影《盗梦空间》,包含电影名、上映年份、导演、演员列表、评分")
# 4. 访问嵌套数据
print(f"电影名: {response.title}")
print(f"上映年份: {response.year}")
print(f"导演: {response.director}")
print(f"演员列表: {response.cast}")
print(f"评分: {response.rating}")





title='盗梦空间' year=2010 director='克里斯托弗·诺兰' cast=[Actor(name='莱昂纳多·迪卡普里奥', role='柯布'), Actor(name='玛丽昂·歌迪亚', role='梅尔'), Actor(name='约瑟夫·高登-莱维特', role='亚瑟'), Actor(name='汤姆·哈迪', role='伊姆斯'), Actor(name='艾伦·佩吉', role='阿丽亚德妮'), Actor(name='渡边谦', role='斋藤'), Actor(name='希里安·墨菲', role='罗伯特·费希尔'), Actor(name='迈克尔·凯恩', role='迈尔斯教授')] rating=9.3
电影名: 盗梦空间
上映年份: 2010
导演: 克里斯托弗·诺兰
演员列表: [Actor(name='莱昂纳多·迪卡普里奥', role='柯布'), Actor(name='玛丽昂·歌迪亚', role='梅尔'), Actor(name='约瑟夫·高登-莱维特', role='亚瑟'), Actor(name='汤姆·哈迪', role='伊姆斯'), Actor(name='艾伦·佩吉', role='阿丽亚德妮'), Actor(name='渡边谦', role='斋藤'), Actor(name='希里安·墨菲', role='罗伯特·费希尔'), Actor(name='迈克尔·凯恩', role='迈尔斯教授')]
评分: 9.3


In [31]:
from pydantic import ValidationError

try:
    response: Movie = structured_model.invoke("请介绍电影《盗梦空间》")
except ValidationError as exc:
    print(exc.errors())
    print(repr(exc.errors()[0]['type']))

[{'type': 'missing', 'loc': ('year',), 'msg': 'Field required', 'input': {'title': '盗梦空间'}, 'url': 'https://errors.pydantic.dev/2.12/v/missing'}, {'type': 'missing', 'loc': ('director',), 'msg': 'Field required', 'input': {'title': '盗梦空间'}, 'url': 'https://errors.pydantic.dev/2.12/v/missing'}, {'type': 'missing', 'loc': ('cast',), 'msg': 'Field required', 'input': {'title': '盗梦空间'}, 'url': 'https://errors.pydantic.dev/2.12/v/missing'}, {'type': 'missing', 'loc': ('rating',), 'msg': 'Field required', 'input': {'title': '盗梦空间'}, 'url': 'https://errors.pydantic.dev/2.12/v/missing'}]
'missing'


说明：LLM 能力有限，复杂嵌套结构可能会出错。所以建议：
嵌套层级 ≤ 3 层

In [ ]:
class Bad(BaseModel):
    user: User
    company: Company
    address: Address
    country: Movie  # 4 层嵌套，容易出错

使用清晰的 description
必要时拆分成多个调用

举例3：

In [32]:
from pydantic import BaseModel
from typing import List


class Aspect(BaseModel):
    """评论维度"""
    name: str = Field(description="维度名称，如：质量、价格、服务")
    score: int = Field(description="评分，1-5")
    comment: str = Field(description="具体评价")


class ProductReview(BaseModel):
    """产品评论分析"""
    overall_sentiment: str = Field(description="整体情感：positive / negative / neutral")
    overall_score: int = Field(description="综合评分，1-5")
    aspects: List[Aspect] = Field(description="各维度评价")
    summary: str = Field(description="一句话总结")


# 创建结构化模型
structured_model = model.with_structured_output(ProductReview)
# 测试
review_text = """
    这款笔记本电脑性能非常强大，运行大型软件毫无压力。
    屏幕色彩鲜艳，看视频很舒服。
    不过价格有点贵，而且风扇噪音较大。
    客服态度很好，物流也快。
    总体来说还是值得购买的。"""

result = structured_model.invoke(f"分析以下产品评论：\n{review_text}")
print(f"整体情感: {result.overall_sentiment}")
print(f"综合评分: {result.overall_score}/5")
print(f"\n各维度评价:")
for aspect in result.aspects:
    print(f"  - {aspect.name}: {aspect.score}/5 - {aspect.comment}")
print(f"\n总结: {result.summary}")


整体情感: positive
综合评分: 4/5

各维度评价:
  - 性能: 5/5 - 性能非常强大，运行大型软件毫无压力
  - 屏幕: 5/5 - 屏幕色彩鲜艳，看视频很舒服
  - 价格: 2/5 - 价格有点贵
  - 噪音: 2/5 - 风扇噪音较大
  - 客服: 5/5 - 客服态度很好
  - 物流: 5/5 - 物流很快

总结: 这款笔记本电脑性能出色、屏幕优质、客服和物流体验好，但价格偏高且风扇噪音较大，整体值得购买。


情况6：限制条件

In [33]:
from pydantic import ValidationError


class User(BaseModel):
    name: str = Field(min_length=2, max_length=20)
    age: int = Field(ge=0, le=150)
    email: str


print("\n有效数据:")
try:
    user = User(name="张三", age=30, email="zhang@example.com")
    print(f"[OK] {user.name}, {user.age}, {user.email}")
except ValidationError as e:
    print(f"[FAIL] {e}")
print("\n无效数据（年龄超出范围）:")
try:
    user = User(name="李四", age=200, email="li@example.com")
    print(f"[OK] {user}")
except ValidationError as e:
    print(f"[FAIL] 验证失败（符合预期）: {e.errors()[0]['msg']}")


有效数据:
[OK] 张三, 30, zhang@example.com

无效数据（年龄超出范围）:
[FAIL] 验证失败（符合预期）: Input should be less than or equal to 150
